In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from tqdm import tqdm
import os
from scipy.stats import norm
from pathlib import Path

C:\Users\Administrator\AppData\Local\Temp\ipykernel_16652\626493030.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [ ]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"

In [2]:
names = locals()

In [ ]:
df_cement_production          = pd.read_csv(DATA / 'production-142.csv', index_col=0)
economies_142                 = df_cement_production.columns.tolist()
stock_growth_of_economies_142 = pd.read_csv(RESULTS/ 'stock-growth-2050.csv', index_col=0)

region_country = pd.read_excel(DATA / 'contents.xlsx', sheet_name='region_division')
region_country.loc[28, 'region'] = 'NA'
region_country.loc[171, 'region'] = 'NA'

In [4]:
def make_uncertain_values(df, std, span, run, seed):
    values = np.array([df]) if type(df) == float else np.array(df)
    if len(values) == 92:
        np.random.seed(seed)
        gg = np.random.normal(values, values * std, size=(run, span))
    elif len(values) == 1:
        np.random.seed(seed)
        gg = np.random.normal(values, values * std, size=(run, span))

    return gg.astype(np.float32) 

In [5]:
def fit_logistic_with_constraints(
                                  hist_values,
                                  future_growth,
                                  start=1931, mid=2022, end=2050,
                                    r_grid=np.linspace(0.05, 0.8, 100),
                                    t0_grid=np.linspace(1950, 2040, 120),
                                    recent_weight=3.0, recent_from=1990):
        
    def logistic_4p(t, A, K, r, t0):
        return A + (K - A) / (1.0 + np.exp(-r * (t - t0)))
    
    def solve_AK_for_endpoints(r, t0, y_mid, y_end):
        L1 = 1.0 / (1.0 + np.exp(-r * (mid - t0)))
        L2 = 1.0 / (1.0 + np.exp(-r * (end - t0)))
        M = np.array([[1 - L1, L1],
                    [1 - L2, L2]], dtype=float)
        y = np.array([y_mid, y_end], dtype=float)
        A, K = np.linalg.solve(M, y)
        return A, K


    hist_years = np.array([t for t in range(start, mid+1)])
    # hist_values = np.array(df_hist.values.tolist())

    y_mid = hist_values[-1]
    y_end = y_mid * future_growth

    w = np.ones_like(hist_years, dtype=float)
    w[hist_years >= recent_from] = recent_weight

    best = {"err": np.inf, "r": None, "t0": None, "A": None, "K": None}
    for r in r_grid:
        for t0 in t0_grid:
            L1 = 1.0 / (1.0 + np.exp(-r * (mid - t0)))
            L2 = 1.0 / (1.0 + np.exp(-r * (end - t0)))
            if abs(L2 - L1) < 1e-6:
                continue

            A, K = solve_AK_for_endpoints(r, t0, y_mid, y_end)
            if not (r > 0 and K > A):
                continue

            y_hat = logistic_4p(hist_years, A, K, r, t0)
            err = np.average((y_hat - hist_values) ** 2, weights=w)
            if err < best["err"]:
                best = {"err": err, "r": r, "t0": t0, "A": A, "K": K}


    r = best["r"]
    t0 = best["t0"]
    A = best["A"]
    K = best["K"]
    years = np.arange(start, end + 1, dtype=float)
    values = logistic_4p(years, A, K, r, t0)
    df_results = pd.Series(values, years)

    return df_results

In [6]:
def cal_dem_rate_matrix(lft_mat, start=1931, mid=2022, end=2050):
    """
    按原 cal_dem_rate1 逻辑改造版本。
    输入：
        lft_mat : (10000, 92) 历史 lifetime 矩阵
    输出：
        sur_rate, dem_rate : (10000, 92, 29)
    """
    n_sim, n_hist = lft_mat.shape
    span = end - mid + 1  # 包含2022→2050

    # 1️⃣ 生成 age 矩阵 (92, 29)
    age_2D = np.array([[j - i for j in range(mid, end + 1)] for i in range(start, mid + 1)], dtype=np.float32)

    # 2️⃣ 扩展 lifetime 与 age
    lft_3D = np.repeat(lft_mat[:, :, None], span, axis=2)  # (10000, 92, 29)
    age_3D = np.broadcast_to(age_2D, (n_sim, n_hist, span))

    # 3️⃣ 计算 future 累积分布函数 (逻辑同 ddd)
    ddd = norm.cdf(age_3D, loc=lft_3D, scale=lft_3D * 0.2).astype(np.float32)

    # 4️⃣ 计算 ...
    gg = np.array([(1 - ddd)[:, :, 0] for i in range(span - 1)]).transpose(1, 2, 0)
    sur_rate = np.where(gg == 0, 0, (1 - ddd)[:, :, 1:]) / np.where(gg == 0, 1, gg)
    dem_rate = np.where(gg == 0, 0, np.diff(ddd, axis=2)) / np.where(gg == 0, 1, gg)
        
    return sur_rate, dem_rate

In [7]:
def cal_dem_rate_matrix2(fut_lft_mat, mid=2022, end=2050):
    span = end - mid
    n_sim, _ = fut_lft_mat.shape
    age_2D_ = np.array([[i-j for i in range(span)] for j in range(span)])
    age_2D = np.where(age_2D_ >= 0, age_2D_, np.nan)
    age_3D = np.broadcast_to(age_2D, (n_sim, span, span)).astype(np.float32)

    lft_3D_ = np.repeat(fut_lft_mat[:, :, None], span, axis=2)
    lft_3D = np.where(age_3D >=0, lft_3D_, np.nan)
    ccc = norm.cdf(age_3D, loc=lft_3D, scale=lft_3D * 0.2).astype(np.float32)

    sur_rate = 1 - ccc
    dem_rate = np.full(sur_rate.shape, np.nan).astype(np.float32)
    dem_rate[:, :, 1:] = np.diff(ccc, axis=2)
    dem_rate = np.where(ccc >= 0, dem_rate, np.nan)
    dem_rate[:, [i for i in range(span)], [i for i in range(span)]] = ccc[:, [i for i in range(28)], [i for i in range(28)]]


    return sur_rate, dem_rate

In [8]:
def process_a_flow(
        country,
        material, 
        app, 
        strength, 
        sur_rate, 
        dem_rate, 
        fut_sur_rate, 
        fut_dem_rate,
        future_stock_trajectory,
        output_dir, 
        n_sim=10000,
        seed=1
        ):
    mid, end = 2022, 2050
    span = end - mid
    # lft = np.vstack([data1[f'lft_{app}'], data2[f'lft_{app}']])
    # fut_lft = np.array([lft[:, -1] * lft_extension for i in range(span)]).T
    # sur_rate, dem_rate = cal_dem_rate_matrix(lft)
    # fut_sur_rate, fut_dem_rate = cal_dem_rate_matrix2(fut_lft)

    data1 = np.load(f'../results/material-flows/historical-uncertainty/{country}-1.npz', allow_pickle=True)
    data2 = np.load(f'../results/material-flows/historical-uncertainty/{country}-2.npz', allow_pickle=True)
    if_developed = region_country[region_country['ISO-3'] == country]['developed'].values[0]
    # std = 0.05 if if_developed == 1 else 0.1
    std = 0.05


    coarse_to_binder_params = {'C15': 4.182638806, 'C16_23': 3.271740668, 'C24_35': 2.584728036, 'C35': 2.471397967}
    fine_to_binder_params   = {'C15': 2.542840011, 'C16_23': 2.855202053, 'C24_35': 2.083908811, 'C35': 1.728617340}


    if material == 'concrete':
        stock_cohort = np.vstack([data1[f'S_binder_for_concrete_{app}_{strength}_cohort'], data2[f'S_binder_for_concrete_{app}_{strength}_cohort']])
        his_stock_annual = np.vstack([data1[f'S_binder_for_concrete_{app}_{strength}_annual'], data2[f'S_binder_for_concrete_{app}_{strength}_annual']])
        # coarse_to_binder = np.vstack([data1[f'coarse_to_binder_{strength}'], data2[f'coarse_to_binder_{strength}']])[:, -1]
        # fine_to_binder = np.vstack([data1[f'fine_to_binder_{strength}'], data2[f'fine_to_binder_{strength}']])[:, -1]
        alpha = coarse_to_binder_params[strength]
        beta = fine_to_binder_params[strength]
        coarse_to_binder = make_uncertain_values(alpha, std, span, n_sim, seed)
        fine_to_binder   = make_uncertain_values(beta,  std, span, n_sim, seed)

    else:
        stock_cohort = np.vstack([data1[f'S_binder_for_mortar_{app}_cohort'], data2[f'S_binder_for_mortar_{app}_cohort']])
        his_stock_annual = np.vstack([data1[f'S_binder_for_mortar_{app}_annual'], data2[f'S_binder_for_mortar_{app}_annual']])
        # fine_to_binder = np.vstack([data1['fine_to_binder_mortar'], data2['fine_to_binder_mortar']])[:, -1]
        fine_to_binder = make_uncertain_values(2.720389651, std, span, n_sim, seed)

    fut_his_stock_array_ = np.array([stock_cohort for i in range(span)]).transpose(1, 2, 0)
    fut_his_stock_array = fut_his_stock_array_ * sur_rate
    fut_his_outflow_array = fut_his_stock_array_ * dem_rate
    fut_his_stock_annual = np.nansum(fut_his_stock_array, axis=1)
    fut_his_outflow_annual = np.nansum(fut_his_outflow_array, axis=1)

    current_stock = his_stock_annual[:, -1]
    future_stock_annual = current_stock[:, None] * future_stock_trajectory[None, :]

    the_gap = future_stock_annual[:, 1:] - fut_his_stock_annual

    fut_fut_stock_array = np.full(shape=(n_sim, span, span), fill_value=np.nan).astype(np.float32)
    fut_fut_outflow_array = np.full(shape=(n_sim, span, span), fill_value=np.nan).astype(np.float32)
    future_inflows = []
    for t in range(span):
        a_inflow = the_gap[:, t] - np.nansum(fut_fut_stock_array[:, :t, t], axis=1)
        fut_fut_stock_array[:, t, :] = a_inflow[:, None] * fut_sur_rate[:, t, :]
        fut_fut_outflow_array[:, t, :] = a_inflow[:, None] * fut_dem_rate[:, t, :]
        future_inflows.append(a_inflow)
    
    fut_fut_stock_annual = np.nansum(fut_fut_stock_array, axis=1)
    fut_fut_outflow_annual = np.nansum(fut_fut_outflow_array, axis=1)

    future_inflows_annual = np.array(future_inflows).T
    future_outflows_annual = fut_his_outflow_annual + fut_fut_outflow_annual

    # future_fine_inflow = future_inflows_annual * fine_to_binder[:, None]
    # future_fine_outflow = future_outflows_annual * fine_to_binder[:, None]

    future_fine_inflow = future_inflows_annual * fine_to_binder
    future_fine_outflow = future_outflows_annual * fine_to_binder


    if material == 'concrete':
        # print(future_inflows_annual.shape)
        # futuer_coarse_inflow = future_inflows_annual * coarse_to_binder[:, None]
        # future_coarse_outflow = future_outflows_annual * coarse_to_binder[:, None]
        futuer_coarse_inflow = future_inflows_annual * coarse_to_binder
        future_coarse_outflow = future_outflows_annual * coarse_to_binder

        np.savez_compressed(os.path.join(output_dir, f'S_binder_for_concrete_{app}_{strength}_his_annual.npz'), array=fut_his_stock_annual)
        np.savez_compressed(os.path.join(output_dir, f'O_binder_for_concrete_{app}_{strength}_his_annual.npz'), array=fut_his_outflow_annual)
        np.savez_compressed(os.path.join(output_dir, f'S_binder_for_concrete_{app}_{strength}_annual.npz'), array=future_stock_annual)
        np.savez_compressed(os.path.join(output_dir, f'I_binder_for_concrete_{app}_{strength}_annual.npz'), array=future_inflows_annual)
        np.savez_compressed(os.path.join(output_dir, f'O_binder_for_concrete_{app}_{strength}_annual.npz'), array=future_outflows_annual)
        np.savez_compressed(os.path.join(output_dir, f'I_coarse_for_concrete_{app}_{strength}_annual.npz'), array=futuer_coarse_inflow)
        np.savez_compressed(os.path.join(output_dir, f'O_coarse_for_concrete_{app}_{strength}_annual.npz'), array=future_coarse_outflow)
        np.savez_compressed(os.path.join(output_dir, f'I_fine_for_concrete_{app}_{strength}_annual.npz'), array=future_fine_inflow)
        np.savez_compressed(os.path.join(output_dir, f'O_fine_for_concrete_{app}_{strength}_annual.npz'), array=future_fine_outflow)
        
    else:
        np.savez_compressed(os.path.join(output_dir, f'S_binder_for_mortar_{app}_his_annual.npz'), array=fut_his_stock_annual)
        np.savez_compressed(os.path.join(output_dir, f'O_binder_for_mortar_{app}_his_annual.npz'), array=fut_his_outflow_annual)
        np.savez_compressed(os.path.join(output_dir, f'S_binder_for_mortar_{app}_annual.npz'), array=future_stock_annual)
        np.savez_compressed(os.path.join(output_dir, f'I_binder_for_mortar_{app}_annual.npz'), array=future_inflows_annual)
        np.savez_compressed(os.path.join(output_dir, f'O_binder_for_mortar_{app}_annual.npz'), array=future_outflows_annual)
        np.savez_compressed(os.path.join(output_dir, f'I_fine_for_mortar_{app}_annual.npz'), array=future_fine_inflow)
        np.savez_compressed(os.path.join(output_dir, f'O_fine_for_mortar_{app}_annual.npz'), array=future_fine_outflow)

In [9]:
variable_list = [
    'I_binder_annual',
    'I_fine_annual',
    'I_coarse_annual',
    'O_binder_annual',
    'O_fine_annual',
    'O_coarse_annual',
    'reuse_factors',
    'recycle_factors',
    'sand_sub_factors',
    'fut_CCR',
    'I_binder_for_concrete_annual_ru',
    'I_binder_for_mortar_annual_ru',
    'I_binder_for_concrete_annual_v',
    'I_binder_for_mortar_annual_v',
    'I_binder_annual_ru',
    'I_binder_annual_v',
    'p_binder_slag',
    'p_binder_FAP',
    'p_binder_in_exportPro',
    'p_binder_in_importPro',
    'p_cement_export',
    'p_cement_import',
    'p_clinker_export',
    'p_clinker_import',
    'p_sand_export',
    'p_sand_import',
    'p_gravel_export',
    'p_gravel_import',
    'p_limestone_export',
    'p_limestone_import',
    'gravel_export',
    'gravel_import',
    'sand_export',
    'sand_import',
    'excess_gravel',
    'excess_sand',
    'gravel_stock',
    'sand_stock',
    'consumed_gravel_stock',
    'consumed_sand_stock',
    'gravel_production',
    'sand_production',
    'reused_and_recycled_gravel',
    'p_reused_gravel',
    'p_recycled_gravel',
    'reused_gravel',
    'recycled_gravel',
    'gravel_extraction',
    'reused_and_recycled_sand',
    'p_reused_sand',
    'p_recycled_sand',
    'reused_sand',
    'recycled_sand',
    'substituted_sand',
    'sand_extraction',
    'all_binder',
    'cement_consumption',
    'cement_production',
    'clinker_consumption',
    'excess_clinker',
    'clinker_stock',
    'clinker_production',
    'r_Ca',
    'RawMat_Ca',
    'limestone_export',
    'limestone_import',
    'excess_limestone',
    'limestone_stock',
    'consumed_limestone_stock',
    'limestone_stock',
    'limestone_extraction',
    'S_coarse_annual',
    'S_fine_annual',
    'S_binder_annual',
    'split_ratio_coarse',
    'split_ratio_fine',
    'split_ratio_RawMat_Ca',
    'gravel_extraction_for_stock_growth',
    'gravel_extraction_for_stock_turnover',
    'sand_extraction_for_stock_growth',
    'sand_extraction_for_stock_turnover',
    'limestone_extraction_for_stock_growth',
    'limestone_extraction_for_stock_turnover'
]

In [10]:
def run_prediction(
    scenario,
    country,
    more_intensive_use=0,
    lifetime_extension=0,
    reuse_rate=0,
    recycle_rate=0,
    sand_sub=0,
    CCR_reduction=0,
    end=2050,
    mid=2022,
    n_sim=10000,
    seed=1
):
    output_dir = os.path.join('..', 'results', 'material-flows', 'future-uncertainty', country, scenario)
    os.makedirs(output_dir, exist_ok=True)

    if_developed = region_country[region_country['ISO-3'] == country]['developed'].values[0]
    std = 0.05 if if_developed == 1 else 0.1

    data1 = np.load(f'../results/material-flows/historical-uncertainty/{country}-1.npz', allow_pickle=True)
    data2 = np.load(f'../results/material-flows/historical-uncertainty/{country}-2.npz', allow_pickle=True)
    
    lft_extension = 1 + lifetime_extension
    span = end - mid

    S_binder_annual = np.vstack([data1['S_binder_annual'], data2['S_binder_annual']])
    historical_stock_trajectory = np.median(S_binder_annual, axis=0) / np.median(S_binder_annual, axis=0)[-1]
    K = stock_growth_of_economies_142.loc[country, 'growth']
    # fitted_stock_trajectory = fit_logistic_growth(stock_growth_of_economies_142.loc[country, 'growth'])
    fitted_stock_trajectory = fit_logistic_with_constraints(historical_stock_trajectory, K)
    future_stock_trajectory_ = fitted_stock_trajectory.loc[mid:end].values
    fut_stock_factor = np.linspace(1, 1 - more_intensive_use, span+1)
    future_stock_trajectory = future_stock_trajectory_ * fut_stock_factor

    for app in ['RES', 'NRES', 'CIV']:
        lft = np.vstack([data1[f'lft_{app}'], data2[f'lft_{app}']])
        fut_lft = np.array([lft[:, -1] * lft_extension for i in range(span)]).T
        sur_rate, dem_rate = cal_dem_rate_matrix(lft)
        fut_sur_rate, fut_dem_rate = cal_dem_rate_matrix2(fut_lft)

        for strength in ['C15', 'C16_23', 'C24_35', 'C35']:
            process_a_flow(
                country,
                'concrete', 
                app, 
                strength,
                sur_rate,
                dem_rate,
                fut_sur_rate,
                fut_dem_rate,
                future_stock_trajectory,
                output_dir)

        binder_stock = np.nansum([np.load(os.path.join(output_dir, f'S_binder_for_concrete_{app}_{strength}_annual.npz'))['array'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)
        binder_inflow = np.nansum([np.load(os.path.join(output_dir, f'I_binder_for_concrete_{app}_{strength}_annual.npz'))['array'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)
        binder_outflow = np.nansum([np.load(os.path.join(output_dir, f'O_binder_for_concrete_{app}_{strength}_annual.npz'))['array'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)
        coarse_inflow = np.nansum([np.load(os.path.join(output_dir, f'I_coarse_for_concrete_{app}_{strength}_annual.npz'))['array'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)
        coarse_outflow = np.nansum([np.load(os.path.join(output_dir, f'O_coarse_for_concrete_{app}_{strength}_annual.npz'))['array'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)
        fine_inflow = np.nansum([np.load(os.path.join(output_dir, f'I_fine_for_concrete_{app}_{strength}_annual.npz'))['array'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)
        fine_outflow = np.nansum([np.load(os.path.join(output_dir, f'O_fine_for_concrete_{app}_{strength}_annual.npz'))['array'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)

        np.savez_compressed(os.path.join(output_dir, f'S_binder_for_concrete_{app}_annual.npz'), array=binder_stock)
        np.savez_compressed(os.path.join(output_dir, f'I_binder_for_concrete_{app}_annual.npz'), array=binder_inflow)
        np.savez_compressed(os.path.join(output_dir, f'O_binder_for_concrete_{app}_annual.npz'), array=binder_outflow)
        np.savez_compressed(os.path.join(output_dir, f'I_coarse_for_concrete_{app}_annual.npz'), array=coarse_inflow)
        np.savez_compressed(os.path.join(output_dir, f'O_coarse_for_concrete_{app}_annual.npz'), array=coarse_outflow)
        np.savez_compressed(os.path.join(output_dir, f'I_fine_for_concrete_{app}_annual.npz'), array=fine_inflow)
        np.savez_compressed(os.path.join(output_dir, f'O_fine_for_concrete_{app}_annual.npz'), array=fine_outflow)

        process_a_flow(
            country,
            'mortar', 
            app,
            '',
            sur_rate,
            dem_rate,
            fut_sur_rate,
            fut_dem_rate,
            future_stock_trajectory,
            output_dir
            )


    for material in ['concrete', 'mortar']:
        binder_stock = np.nansum([np.load(os.path.join(output_dir, f'S_binder_for_{material}_{app}_annual.npz'))['array'] for app in ['RES', 'NRES', 'CIV']], axis=0).astype(np.float32)
        binder_inflow = np.nansum([np.load(os.path.join(output_dir, f'I_binder_for_{material}_{app}_annual.npz'))['array'] for app in ['RES', 'NRES', 'CIV']], axis=0).astype(np.float32)
        binder_outflow = np.nansum([np.load(os.path.join(output_dir, f'O_binder_for_{material}_{app}_annual.npz'))['array'] for app in ['RES', 'NRES', 'CIV']], axis=0).astype(np.float32)
        fine_inflow = np.nansum([np.load(os.path.join(output_dir, f'I_fine_for_{material}_{app}_annual.npz'))['array'] for app in ['RES', 'NRES', 'CIV']], axis=0).astype(np.float32)
        fine_outflow = np.nansum([np.load(os.path.join(output_dir, f'O_fine_for_{material}_{app}_annual.npz'))['array'] for app in ['RES', 'NRES', 'CIV']], axis=0).astype(np.float32)
        
        np.savez_compressed(os.path.join(output_dir, f'S_binder_for_{material}_annual.npz'), array=binder_stock)
        np.savez_compressed(os.path.join(output_dir, f'I_binder_for_{material}_annual.npz'), array=binder_inflow)
        np.savez_compressed(os.path.join(output_dir, f'O_binder_for_{material}_annual.npz'), array=binder_outflow)
        np.savez_compressed(os.path.join(output_dir, f'I_fine_for_{material}_annual.npz'), array=fine_inflow)
        np.savez_compressed(os.path.join(output_dir, f'O_fine_for_{material}_annual.npz'), array=fine_outflow)

        if material == 'concrete':
            coarse_inflow = np.nansum([np.load(os.path.join(output_dir, f'I_coarse_for_{material}_{app}_annual.npz'))['array'] for app in ['RES', 'NRES', 'CIV']], axis=0).astype(np.float32)
            coarse_outflow = np.nansum([np.load(os.path.join(output_dir, f'O_coarse_for_{material}_{app}_annual.npz'))['array'] for app in ['RES', 'NRES', 'CIV']], axis=0).astype(np.float32)

            np.savez_compressed(os.path.join(output_dir, f'I_coarse_for_{material}_annual.npz'), array=coarse_inflow)
            np.savez_compressed(os.path.join(output_dir, f'O_coarse_for_{material}_annual.npz'), array=coarse_outflow)
    
    I_binder_for_concrete_annual = np.load(os.path.join(output_dir, 'I_binder_for_concrete_annual.npz'))['array']
    I_binder_for_mortar_annual = np.load(os.path.join(output_dir, 'I_binder_for_mortar_annual.npz'))['array']
    I_binder_annual = I_binder_for_concrete_annual + I_binder_for_mortar_annual
    I_fine_annual = np.load(os.path.join(output_dir, 'I_fine_for_concrete_annual.npz'))['array'] + np.load(os.path.join(output_dir, 'I_fine_for_mortar_annual.npz'))['array']
    I_coarse_annual = np.load(os.path.join(output_dir, 'I_coarse_for_concrete_annual.npz'))['array']
    O_binder_annual = np.load(os.path.join(output_dir, 'O_binder_for_concrete_annual.npz'))['array'] + np.load(os.path.join(output_dir, 'O_binder_for_mortar_annual.npz'))['array']
    O_fine_annual = np.load(os.path.join(output_dir, 'O_fine_for_concrete_annual.npz'))['array'] + np.load(os.path.join(output_dir, 'O_fine_for_mortar_annual.npz'))['array']
    O_coarse_annual = np.load(os.path.join(output_dir, 'O_coarse_for_concrete_annual.npz'))['array']

    reuse_factors = np.tile(np.linspace(0, reuse_rate, span+1)[1:], (n_sim, 1))
    recycle_factors = np.tile(np.linspace(0, recycle_rate, span+1)[1:], (n_sim, 1))
    sand_sub_factors = np.tile(np.linspace(0, sand_sub, span+1)[1:], (n_sim, 1))

    CCR_factors = np.tile(np.linspace(1, 1 - CCR_reduction, span+1)[1:], (n_sim, 1))
    his_CCR = np.vstack([data1['CCR'], data2['CCR']])
    fut_CCR = CCR_factors * his_CCR[:, -1][:, None]

    I_binder_for_concrete_annual_ru = np.load(os.path.join(output_dir, 'O_binder_for_concrete_annual.npz'))['array'] * reuse_factors 
    I_binder_for_mortar_annual_ru = np.load(os.path.join(output_dir, 'O_binder_for_mortar_annual.npz'))['array'] * reuse_factors 
    I_binder_for_concrete_annual_v = I_binder_for_concrete_annual - I_binder_for_concrete_annual_ru 
    I_binder_for_mortar_annual_v = I_binder_for_mortar_annual - I_binder_for_mortar_annual_ru
    I_binder_annual_ru = I_binder_for_concrete_annual_ru + I_binder_for_mortar_annual_ru
    I_binder_annual_v =  I_binder_for_concrete_annual_v + I_binder_for_mortar_annual_v

    p_binder_slag = np.vstack([data1['binder_slag'], data2['binder_slag']])[:, -1]
    p_binder_FAP = np.vstack([data1['binder_FAP'], data2['binder_FAP']])[:, -1]

    p_binder_in_exportPro = np.vstack([data1['binder_in_exportPro'], data2['binder_in_exportPro']])[:, -1] / np.vstack([data1['binder_for_concrete'], data2['binder_for_concrete']])[:, -1]
    p_binder_in_importPro = np.vstack([data1['binder_in_importPro'], data2['binder_in_importPro']])[:, -1] / np.vstack([data1['binder_for_concrete'], data2['binder_for_concrete']])[:, -1]

    his_cement_consumption = np.vstack([data1['cement_consumption'], data2['cement_consumption']])
    his_cement_export = np.vstack([data1['cement_export'], data2['cement_export']])
    his_cement_import = np.vstack([data1['cement_import'], data2['cement_import']])

    his_clinker_consumption = np.vstack([data1['clinker_consumption'], data2['clinker_consumption']])
    his_clinker_export = np.vstack([data1['clinker_export'], data2['clinker_export']])
    his_clinker_import = np.vstack([data1['clinker_import'], data2['clinker_import']])

    his_fine_total = np.vstack([data1['fine_total'], data2['fine_total']])
    his_sand_export = np.vstack([data1['sand_export'], data2['sand_export']])
    his_sand_import = np.vstack([data1['sand_import'], data2['sand_import']])

    his_coarse_total = np.vstack([data1['coarse_total'], data2['coarse_total']])
    his_gravel_export = np.vstack([data1['gravel_export'], data2['gravel_export']])
    his_gravel_import = np.vstack([data1['gravel_import'], data2['gravel_import']])

    his_RawMat_Ca = np.vstack([data1['RawMat_Ca'], data2['RawMat_Ca']])
    his_limestone_export = np.vstack([data1['limestone_export'], data2['limestone_export']])
    his_limestone_import = np.vstack([data1['limestone_import'], data2['limestone_import']])

    p_cement_export = np.divide(his_cement_export[:, -1], his_cement_consumption[:, -1], out=np.zeros_like(his_cement_export[:, -1], dtype=np.float32), where=his_cement_consumption[:, -1] != 0)
    p_cement_import = np.divide(his_cement_import[:, -1], his_cement_consumption[:, -1], out=np.zeros_like(his_cement_import[:, -1], dtype=np.float32), where=his_cement_consumption[:, -1] != 0)

    p_clinker_export = np.divide(his_clinker_export[:, -1], his_clinker_consumption[:, -1], out=np.zeros_like(his_clinker_export[:, -1], dtype=np.float32), where=his_clinker_consumption[:, -1] != 0)
    p_clinker_import = np.divide(his_clinker_import[:, -1], his_clinker_consumption[:, -1], out=np.zeros_like(his_clinker_import[:, -1], dtype=np.float32), where=his_clinker_consumption[:, -1] != 0)

    p_sand_export = np.divide(his_sand_export[:, -1], his_fine_total[:, -1], out=np.zeros_like(his_sand_export[:, -1], dtype=np.float32), where=his_fine_total[:, -1] != 0)
    p_sand_import = np.divide(his_sand_import[:, -1], his_fine_total[:, -1], out=np.zeros_like(his_sand_import[:, -1], dtype=np.float32), where=his_fine_total[:, -1] != 0)

    p_gravel_export = np.divide(his_gravel_export[:, -1], his_coarse_total[:, -1], out=np.zeros_like(his_gravel_export[:, -1], dtype=np.float32), where=his_coarse_total[:, -1] != 0)
    p_gravel_import = np.divide(his_gravel_import[:, -1], his_coarse_total[:, -1], out=np.zeros_like(his_gravel_import[:, -1], dtype=np.float32), where=his_coarse_total[:, -1] != 0)

    p_limestone_export = np.divide(his_limestone_export[:, -1], his_RawMat_Ca[:, -1], out=np.zeros_like(his_limestone_export[:, -1], dtype=np.float32), where=his_RawMat_Ca[:, -1] != 0)
    p_limestone_import = np.divide(his_limestone_import[:, -1], his_RawMat_Ca[:, -1], out=np.zeros_like(his_limestone_import[:, -1], dtype=np.float32), where=his_RawMat_Ca[:, -1] != 0)

    gravel_production_ = I_coarse_annual / (1 + p_gravel_export - p_gravel_import)[:, None]
    sand_production_ = I_fine_annual / (1 + p_sand_export - p_sand_import)[:, None]

    gravel_export = I_coarse_annual * p_gravel_export[:, None]
    gravel_import = I_coarse_annual * p_gravel_import[:, None]

    sand_export = I_fine_annual * p_sand_export[:, None]
    sand_import = I_fine_annual * p_sand_import[:, None]

    excess_gravel = np.where(gravel_production_ < 0 , -gravel_production_, 0)
    excess_sand = np.where(sand_production_ < 0 , -sand_production_, 0)

    his_gravel_stock = np.vstack([data1['gravel_stock'], data2['gravel_stock']])
    his_sand_stock = np.vstack([data1['sand_stock'], data2['sand_stock']])
    his_limestone_stock = np.vstack([data1['limestone_stock'], data2['limestone_stock']])

    gravel_stock = excess_gravel.cumsum(axis=1) + his_gravel_stock[:, -1][:, None]
    sand_stock = excess_sand.cumsum(axis=1) + his_sand_stock[:, -1][:, None]

    consumed_gravel_stock = np.where((gravel_stock > 0) & (gravel_production_ > 0), np.where(gravel_production_ <= gravel_stock, gravel_production_, gravel_stock), 0)
    consumed_sand_stock = np.where((sand_stock > 0) & (sand_production_ > 0), np.where(sand_production_ <= sand_stock, sand_production_, sand_stock), 0)

    gravel_production = np.where(gravel_production_ >= consumed_gravel_stock, gravel_production_ - consumed_gravel_stock, 0)
    sand_production = np.where(sand_production_ >= consumed_sand_stock, sand_production_ - consumed_sand_stock, 0)

    reused_gravel_ = O_coarse_annual * reuse_factors
    recycled_gravel_ = O_coarse_annual * recycle_factors
    reused_and_recycled_gravel_ = reused_gravel_ + recycled_gravel_

    reused_and_recycled_gravel = np.where(gravel_production >= reused_and_recycled_gravel_, reused_and_recycled_gravel_, gravel_production)

    p_reused_gravel = np.divide(reused_gravel_, reused_and_recycled_gravel_, out=np.zeros_like(reused_gravel_, dtype=np.float32), where=reused_and_recycled_gravel_ != 0)
    p_recycled_gravel = np.divide(recycled_gravel_, reused_and_recycled_gravel_, out=np.zeros_like(recycled_gravel_, dtype=np.float32), where=reused_and_recycled_gravel_ != 0)
    reused_gravel = reused_and_recycled_gravel * p_reused_gravel
    recycled_gravel = reused_and_recycled_gravel * p_recycled_gravel
    gravel_extraction = gravel_production - reused_and_recycled_gravel

    reused_sand_ = O_fine_annual * reuse_factors
    recycled_sand_ = O_fine_annual * recycle_factors
    reused_and_recycled_sand_ = reused_sand_ + recycled_sand_ 
    reused_and_recycled_sand = np.where(sand_production >= reused_and_recycled_sand_, reused_and_recycled_sand_, sand_production)

    p_reused_sand = np.divide(reused_sand_, reused_and_recycled_sand_, out=np.zeros_like(reused_sand_, dtype=np.float32), where=reused_and_recycled_sand_ != 0)
    p_recycled_sand = np.divide(recycled_sand_, reused_and_recycled_sand_, out=np.zeros_like(recycled_sand_, dtype=np.float32), where=reused_and_recycled_sand_ != 0)
    reused_sand = reused_and_recycled_sand * p_reused_sand
    recycled_sand = reused_and_recycled_sand * p_recycled_sand
    substituted_sand = (sand_production - reused_and_recycled_sand) * sand_sub_factors
    sand_extraction = sand_production - (reused_and_recycled_sand + substituted_sand)

    all_binder = I_binder_for_concrete_annual_v * (1 - p_binder_in_importPro + p_binder_in_exportPro)[:, None] + I_binder_for_mortar_annual_v
    cement_consumption = all_binder / (1 + p_binder_slag + p_binder_FAP)[:, None]
    cement_production = cement_consumption / (1 - p_cement_export + p_cement_import)[:, None]
    clinker_consumption = cement_production * fut_CCR
    clinker_production_ = clinker_consumption / (1 - p_clinker_export + p_clinker_import)[:, None]
    excess_clinker = np.where(clinker_production_ < 0, -clinker_production_, 0)
    hh = excess_clinker.cumsum(axis=0)
    clinker_stock = hh.copy()
    clinker_stock[3:] = hh[3:] - hh[:-3]
    clinker_production = np.where(clinker_production_ >= clinker_stock, clinker_production_ - clinker_stock, 0)

    # r_Ca = np.vstack([data1['r_Ca'], data2['r_Ca']])[:, -1]
    r_Ca = make_uncertain_values(1.45402, std, span, n_sim, seed)
    RawMat_Ca = clinker_production * r_Ca
    limestone_ = RawMat_Ca / (1 - p_limestone_export + p_limestone_import)[:, None]
    limestone_export = RawMat_Ca * p_limestone_export[:, None]
    limestone_import = RawMat_Ca * p_limestone_import[:, None]
    excess_limestone = np.where(limestone_ < 0, -limestone_, 0)
    limestone_stock = excess_limestone.cumsum(axis=1) + his_limestone_stock[:, -1][:, None]
    consumed_limestone_stock = np.where((limestone_stock > 0) & (limestone_ > 0), np.where(limestone_ <= limestone_stock, limestone_, limestone_stock), 0)
    limestone_stock = limestone_stock - consumed_limestone_stock
    limestone_extraction = np.where(limestone_ >= consumed_limestone_stock, limestone_ - consumed_limestone_stock, 0)

    
    S_coarse_annual = (np.vstack([data1['S_coarse_annual'], data2['S_coarse_annual']])[:, -1] * future_stock_trajectory[:, None]).T
    S_fine_annual = (np.vstack([data1['S_fine_annual'], data2['S_fine_annual']])[:, -1] * future_stock_trajectory[:, None]).T
    S_binder_annual = (np.vstack([data1['S_binder_annual'], data2['S_binder_annual']])[:, -1] * future_stock_trajectory[:, None]).T

    S_coarse_growth = np.diff(S_coarse_annual, axis=1)
    S_fine_growth = np.diff(S_fine_annual, axis=1)
    S_binder_growth = np.diff(S_binder_annual, axis=1)

    split_ratio_coarse = O_coarse_annual / I_coarse_annual
    split_ratio_fine = O_coarse_annual / I_fine_annual
    split_ratio_RawMat_Ca = O_binder_annual / I_binder_annual

    gravel_extraction_for_stock_growth = (1 - split_ratio_coarse) * gravel_extraction
    sand_extraction_for_stock_growth = (1 - split_ratio_fine) * sand_extraction
    limestone_extraction_for_stock_growth = (1 - split_ratio_RawMat_Ca) * limestone_extraction

    gravel_extraction_for_stock_turnover = split_ratio_coarse * gravel_extraction
    sand_extraction_for_stock_turnover = split_ratio_fine * sand_extraction
    limestone_extraction_for_stock_turnover = split_ratio_RawMat_Ca * limestone_extraction
    
    local_vars = locals()
    for var in variable_list:
        an_array = local_vars[var].astype(np.float32)
        output_path = os.path.join(output_dir, f'{var}.npz')
        np.savez_compressed(output_path, array=an_array)

In [ ]:
for country in tqdm(['CHN']):
    run_prediction(
        'BAU', 
        country, 
        more_intensive_use=0.00,
        lifetime_extension=0.0,
        reuse_rate=0.0,
        recycle_rate=0.0,
        sand_sub=0.0,
        CCR_reduction=0.0
    )

In [ ]:
for country in tqdm(economies_142):
    run_prediction(
        'BAU', 
        country, 
        more_intensive_use=0.00,
        lifetime_extension=0.0,
        reuse_rate=0.0,
        recycle_rate=0.0,
        sand_sub=0.0,
        CCR_reduction=0.0
    )

In [ ]:
for country in tqdm(economies_142):
    run_prediction(
        'CE', 
        country, 
        more_intensive_use=0.05,
        lifetime_extension=0.50,
        reuse_rate=0.1,
        recycle_rate=0.2,
        sand_sub=0.2,
        CCR_reduction=0.25
    )

In [ ]:
for country in tqdm(economies_142):
    run_prediction(
        'MIU', 
        country, 
        more_intensive_use=0.05,
        lifetime_extension=0.0,
        reuse_rate=0.0,
        recycle_rate=0.0,
        sand_sub=0.0,
        CCR_reduction=0.0
    )

In [ ]:
for country in tqdm(economies_142):
    run_prediction(
        'LE', 
        country, 
        more_intensive_use=0.00,    
        lifetime_extension=0.50,
        reuse_rate=0.0,
        recycle_rate=0.0,
        sand_sub=0.0,
        CCR_reduction=0.0
    )

In [ ]:
for country in tqdm(economies_142):
    run_prediction(
        'RE', 
        country, 
        more_intensive_use=0.00,
        lifetime_extension=0.00,
        reuse_rate=0.1,
        recycle_rate=0.0,
        sand_sub=0.0,
        CCR_reduction=0.0
    )

In [ ]:
for country in tqdm(economies_142):
    run_prediction(
        'RC', 
        country, 
        more_intensive_use=0.00,
        lifetime_extension=0.00,
        reuse_rate=0.0,
        recycle_rate=0.2,
        sand_sub=0.0,
        CCR_reduction=0.0
    )

In [ ]:
for country in tqdm(economies_142):
    run_prediction(
        'SS', 
        country, 
        more_intensive_use=0.00,
        lifetime_extension=0.00,
        reuse_rate=0.0,
        recycle_rate=0.0,
        sand_sub=0.2,
        CCR_reduction=0.0
    )

In [ ]:
for country in tqdm(economies_142):
    run_prediction(
        'CCR', 
        country, 
        more_intensive_use=0.00,
        lifetime_extension=0.00,
        reuse_rate=0.0,
        recycle_rate=0.0,
        sand_sub=0.0,
        CCR_reduction=0.25
    )